## Trasformazioni di Householder

Le trasformazioni di Householder sono matrici simmetriche e ortogonali, utilizzate per ruotare un vettore su un asse cartesiano.

**Ricetta**: dato $a\in{\mathbb R}^n$, si definisce la matrice di Householder $U$ nel modo seguente

1. $\sigma = \|a\|$
2. $v = a + \sigma e_1$, dove $e_1$ è il primo vettore della base canonica
3. $\alpha =\frac 1 2 \|v\|^2$
4. $U = I-\frac{1}{\alpha}v v^T$

La matrice così costruita è ortogonale, simmetrica e ha la proprietà di ruotare il vettore $a$ portandolo lungo il primo asse cartesiano, ossia

$$Ua = -\sigma e_1 = \begin{pmatrix} -\|\sigma\|\\0\\\vdots\\ 0
\end{pmatrix} $$

In [6]:
import numpy as np

n = 10
a = np.random.randn(n)

# =========================================
# Trasformazione di Householder (1ª riflessione)
# 
# Obiettivo: azzerare tutte le componenti di a
#            tranne la prima, preservando la norma
# =========================================

# 1) Norma del vettore originale
sigma = np.linalg.norm(a)
print('Norma vettore originale =', sigma)

# 2) Costruzione del vettore di Householder v
v = a.copy()

# Si usa sigma per costruire la riflessione stabile numericamente
# (evita cancellazione numerica)
v[0] += sigma

# v come vettore colonna
v = v.reshape(n, 1)

# 3) Costante di normalizzazione
alpha = (v.T @ v) / 2

# 4) Matrice di Householder
# H = I - 2 * (v v^T) / (v^T v)
H = np.eye(n) - (v @ v.T) / alpha

# =========================================
# Applicazione della trasformazione
# =========================================

print('Vettore riflesso =\n', H @ a)

# Output atteso:
# [ -||a||, 0, 0, ..., 0 ]
# (zeri numerici ~ precisione macchina)

# =========================================
# Proprietà della matrice di Householder
# =========================================

# Ortogonalità: H^T H = I
print('Ortogonalità:', np.linalg.norm(np.eye(n) - H @ H.T))

# Simmetria: H = H^T
print('Simmetria:', np.linalg.norm(H - H.T))

# Auto-inversa: H^-1 = H
print('Auto-inversa:', np.linalg.norm(H - np.linalg.inv(H)))

Norma vettore originale = 3.156044243008427
Vettore riflesso =
 [-3.15604424e+00  2.77555756e-17  2.08166817e-17  4.85722573e-17
  1.80411242e-16 -4.85722573e-17  2.77555756e-17  2.77555756e-17
  0.00000000e+00  1.11022302e-16]
Ortogonalità: 4.5696658861166e-16
Simmetria: 0.0
Auto-inversa: 4.3575756499965556e-16


## Complessità computazionale effettiva nel calcolo del prodotto di un vettore per una matrice di Householder

Una volta definita una trasformazione di Householder associata ad un vettore $v$ con la formula $U=I-\frac{1}{\alpha}vv^T$, se si deve calcolare un prodotto $Uy$, dove $y$ è un qualsiasi vettore in $\mathbb R^n$, l'algoritmo più efficiente si ottiene applicando prima la proprietà distributiva e poi la associativa

$$Uy =\left(I-\frac{1}{\alpha}vv^T\right)y = y-\frac{1}{\alpha}v(v^Ty)  $$

INPUT: $v,y$
1. $\alpha \leftarrow \frac{1}{2} \|v\|^2$
2. $p \leftarrow v^Ty$
3. $z \leftarrow y - v(p/\alpha)$

OUTPUT: $z=Uy$

Calcolato in questo modo, il prodotto $Uy$ ha complessità proporzionale a $n$ invece che a $n^2$

## Stabilità delle fattorizzazioni: esperimento con la matrice di Wilkinson

Consideriamo la seguente matrice $n\times n$ (matrice di Wilkinson)

$$ W = \begin{pmatrix}
1 &   & & & & & 1\\
-1& 1 & & & & & 1\\
-1&-1& 1& & & & 1\\
\vdots & & &\ddots & & &\vdots\\
-1& -1 & -1 &        && 1& 1\\
-1& -1&-1 & \cdots & \cdots &-1&1
\end{pmatrix}$$

* Scrivere una function che, dato un intero $n$, restituisce la matrice di Wilkinson di dimensione $n$.
* Costruire un problema test relativo alla matrice di Wilkinson di dimensione 10 che abbia come soluzione il vettore di tutti 1
Calcolare la soluzione del sistema con la funzione linalg.solve e confrontare la soluzione calcolata con la soluzione esatta.
* Ripetere l'esperimento con $n=60$
* Ricalcolare la matrice con $n=10$, applicare la fattorizzazione $LU$ di Gauss e stampare la matrice $U$
* Risolvere il sistema lineare iniziale ($n=60$) mediante la fattorizzazione $QR$ e confrontare con la soluzione esatta


In [7]:
import numpy as np
from scipy.linalg import lu

# 1) Costruzione della matrice di Wilkinson
def wilkinson(n: int):
    # Parte triangolare inferiore (-1 sotto diagonale)
    W = np.tril(-np.ones((n, n), dtype=float), -1)
    
    # Aggiungo la diagonale principale
    W += np.eye(n)
    
    # Ultima colonna (eccetto ultima riga)
    W[:-1, -1] = 1
    
    return W


# 2) Generazione del problema test
def gen_problem(n: int):
    W = wilkinson(n)     # matrice del sistema
    x = np.ones(n)       # soluzione esatta
    b = W @ x            # termine noto
    return W, x, b


# 3) Risoluzione e analisi errore
n = 100
W, x, b = gen_problem(n)

# Risoluzione con solver numerico
s = np.linalg.solve(W, b)

print(f'(n={n}) soluzione numerica s =\n', s)

# Errore rispetto alla soluzione esatta
err = np.linalg.norm(x - s)
print(f'(n={n}) errore ||x - s|| =', err)

# Nota: la matrice di Wilkinson è mal condizionata,
# quindi l’errore può crescere con n.


# 4) Fattorizzazione LU e analisi struttura
n = 10
W, x, b = gen_problem(n)

# Fattorizzazione LU con pivoting
P, L, U = lu(W.copy())

print(f'(n={n}) matrice U =\n', U)

# Verifica della fattorizzazione
print(f'(n={n}) errore ||PA - LU|| =', np.linalg.norm(P @ W - L @ U))


# Osservazioni:
# - U è triangolare superiore.
# - Per la matrice di Wilkinson, gli elementi possono crescere rapidamente
#   (fenomeno di crescita), causando instabilità numerica.
# - La fase più critica è la sostituzione all’indietro.

(n=100) soluzione numerica s =
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 1.]
(n=100) errore ||x - s|| = 6.782329983125268
(n=10) matrice U =
 [[  1.   0.   0.   0.   0.   0.   0.   0.   0.   1.]
 [  0.   1.   0.   0.   0.   0.   0.   0.   0.   2.]
 [  0.   0.   1.   0.   0.   0.   0.   0.   0.   4.]
 [  0.   0.   0.   1.   0.   0.   0.   0.   0.   8.]
 [  0.   0.   0.   0.   1.   0.   0.   0.   0.  16.]
 [  0.   0.   0.   0.   0.   1.   0.   0.   0.  32.]
 [  0.   0.   0.   0.   0.   0.   1.   0.   0.  64.]
 [  0.   0.   0.   0.   0.   0.   0.   1.   0. 128.]
 [  0.   0.   0.   0.   0.   0.   0.   0.   1. 256.]
 [  0.   0.   0.   0.   0.   0.   0.   0.   0. 512.]]
(n=10) errore ||PA - LU|| = 0.0


In [8]:
from scipy.linalg import qr, solve_triangular

# 5) Risoluzione tramite fattorizzazione QR
n = 100
W, x, b = gen_problem(n)

# Soluzione di riferimento (LU / Gauss)
s = np.linalg.solve(W, b)

# Fattorizzazione QR: A = QR
Q, R = qr(W)

# Calcolo del termine noto trasformato: Q^T b
y = Q.T @ b

# Risoluzione del sistema triangolare superiore Rx = y
x_qr = solve_triangular(R, y)

# Errori
err_gauss = np.linalg.norm(x - s)
err_qr = np.linalg.norm(x - x_qr)

print(f'(n={n}) errore Gauss =', err_gauss)
print(f'(n={n}) errore QR    =', err_qr)


# Osservazioni:
# - QR evita i problemi di instabilità legati al pivoting.
# - L’errore con QR è tipicamente vicino alla precisione di macchina.
# - Con matrici mal condizionate (come Wilkinson), QR è più affidabile.

(n=100) errore Gauss = 6.782329983125268
(n=100) errore QR    = 1.756825768425735e-13
